# SpectralRecon Performance Benchmark

This notebook demonstrates the performance improvement achieved by using `Numba` JIT compilation and parallelization in `batch_numba.py` compared to the pure-Numpy implementation in `batch_recon.py`.

In [ ]:
import numpy as np
from skimage import io
import time
import matplotlib.pyplot as plt

from batch_recon import SpectralRecon
from batch_numba import SpectralReconNumba

## 1. Create a Synthetic Test Image
To ensure the benchmark is reproducible and doesn't require massive external files, we generate a synthetic spinning-disk spectral image pattern.

In [ ]:
height, width = 2048, 2048
cols = np.arange(width)
rows = np.arange(height)[:, None]

period = 50
curvature = 0.0001 * (cols - width/2)**2
phase = rows - curvature

# Create dark and light bands
synthetic_img = 1000 + 500 * np.sin(2 * np.pi * phase / period)
synthetic_img += np.random.normal(0, 50, (height, width))
synthetic_img = synthetic_img.astype(np.float32)

print(synthetic_img.shape)

io.imsave('synthetic_test.tiff', synthetic_img.astype(np.uint16))

plt.figure(figsize=(10, 5))
plt.imshow(synthetic_img[500:1000, 500:1000], cmap='gray')
plt.title('Synthetic Spectral Image Region')
plt.show()

## 2. Initialize Analyzers and Run Numba Warmup
The first execution of Numba code includes compilation time. We run a warmup pass to ensure fair benchmarking.

In [ ]:
analyzer_numpy = SpectralRecon(custom_lines_num=True, lines_num=10, precise_allocation=True)
analyzer_numba = SpectralReconNumba(custom_lines_num=True, lines_num=10, precise_allocation=True)

print("Running Numba warmup...")
res_warmup = analyzer_numba.process_single('synthetic_test.tiff')
print("Warmup complete!")

## 3. Benchmark `process_single`
Comparing single-image reconstruction speed.

In [ ]:
print("--- Standard Numpy Implementation ---")
%time analyzer_numpy.process_single('synthetic_test.tiff')

print("\n--- Numba JIT + Parallel Implementation ---")
%time analyzer_numba.process_single('synthetic_test.tiff')

## 4. Benchmark `process_batch`
Testing accumulation of 5 frames.

In [ ]:
batch_paths = ['synthetic_test.tiff'] * 2

print("--- Standard Numpy Batch ---")
%time analyzer_numpy.process_batch(batch_paths, method='max')

print("\n--- Numba Batch ---")
%time analyzer_numba.process_batch(batch_paths, method='max')

## 5. Verify Output Correctness
Confirming both implementations generate identical analytical outputs.

In [ ]:
res_raw = analyzer_numpy.process_single('synthetic_test.tiff')
res_jit = analyzer_numba.process_single('synthetic_test.tiff')

np.testing.assert_allclose(res_raw.spectral_img, res_jit.spectral_img, rtol=1e-5, atol=1e-5)
print("\u2705 Outputs match perfectly!")